# **Big Data Concepts and PySpark Processing**

**Install PySpark**

In [1]:
!pip install pyspark

**Create PySpark Session**

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BigDataProject") \
    .getOrCreate()

**Upload CSV**

In [3]:
from google.colab import files

uploaded = files.upload()

Saving messy_sales_data.csv to messy_sales_data.csv


**Read CSV Using PySpark**

In [4]:
df = spark.read.csv(
    "messy_sales_data.csv",
    header=True,
    inferSchema=True
)

In [5]:
df.show()

+--------+-------------+----------+-----------+--------+----------+----------+---------+-----------+
|order_id|customer_name|   product|   category|quantity|unit_price|order_date|     city|  sales_rep|
+--------+-------------+----------+-----------+--------+----------+----------+---------+-----------+
|    1001| Ramesh Kumar|    Laptop|Electronics|       2|     45000|2024-01-05|   Mumbai|Anil Sharma|
|    1002|   Priya Nair|      NULL|Electronics|       1|     15000|2024-01-07|    Delhi| Sunita Rao|
|    1003|   AMIT VERMA|  Keyboard|Accessories|       3|      1200|2024-01-08|Bangalore|Anil Sharma|
|    1004| Sunita Patel|   Monitor|Electronics|    NULL|     22000|2024-01-10|  Chennai| Ravi Kumar|
|    1005| Ramesh Kumar|    Laptop|Electronics|       2|     45000|2024-01-05|   Mumbai|Anil Sharma|
|    1006|  kiran mehta|     Mouse|Accessories|      10|       800|07-01-2024|     Pune| Sunita Rao|
|    1007| Deepak Singh|Headphones|Electronics|       2|      3500|2024-01-12|Hyderabad| Ra

In [6]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- city: string (nullable = true)
 |-- sales_rep: string (nullable = true)



In [7]:
df.count()

30

**Data Exploration**

In [8]:
print(df.columns)

['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


In [9]:
df.show(10)

+--------+-------------+----------+-----------+--------+----------+----------+---------+-----------+
|order_id|customer_name|   product|   category|quantity|unit_price|order_date|     city|  sales_rep|
+--------+-------------+----------+-----------+--------+----------+----------+---------+-----------+
|    1001| Ramesh Kumar|    Laptop|Electronics|       2|     45000|2024-01-05|   Mumbai|Anil Sharma|
|    1002|   Priya Nair|      NULL|Electronics|       1|     15000|2024-01-07|    Delhi| Sunita Rao|
|    1003|   AMIT VERMA|  Keyboard|Accessories|       3|      1200|2024-01-08|Bangalore|Anil Sharma|
|    1004| Sunita Patel|   Monitor|Electronics|    NULL|     22000|2024-01-10|  Chennai| Ravi Kumar|
|    1005| Ramesh Kumar|    Laptop|Electronics|       2|     45000|2024-01-05|   Mumbai|Anil Sharma|
|    1006|  kiran mehta|     Mouse|Accessories|      10|       800|07-01-2024|     Pune| Sunita Rao|
|    1007| Deepak Singh|Headphones|Electronics|       2|      3500|2024-01-12|Hyderabad| Ra

**Data Cleaning**

In [10]:
from pyspark.sql.functions import col, when, count

df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

+--------+-------------+-------+--------+--------+----------+----------+----+---------+
|order_id|customer_name|product|category|quantity|unit_price|order_date|city|sales_rep|
+--------+-------------+-------+--------+--------+----------+----------+----+---------+
|       0|            2|      1|       1|       3|         0|         0|   0|        0|
+--------+-------------+-------+--------+--------+----------+----------+----+---------+



In [11]:
df = df.fillna({
    "product": "Unknown"
})

In [12]:
df = df.fillna({
    "quantity": 1
})

In [13]:
df = df.dropDuplicates()

In [14]:
df.count()

30

**Create Revenue Column**

In [15]:
from pyspark.sql.functions import col

df = df.withColumn(
    "revenue",
    col("quantity") * col("unit_price")
)

In [16]:
df.select(
    "order_id",
    "city",
    "revenue"
).show()

+--------+------------------+-------+
|order_id|              city|revenue|
+--------+------------------+-------+
|    1014|             Kochi|  10500|
|    1029|         Allahabad|   6000|
|    1021|        Chandigarh|   7000|
|    1012|         Bangalore|   4800|
|    1009|           Kolkata|  45000|
|    1023|           Kolkata|  45000|
|    1010|           Chennai|   6000|
|    1007|         Hyderabad|   7000|
|    1028|Thiruvananthapuram| 135000|
|    1006|              Pune|   8000|
|    1025|           Kolkata|   4800|
|    1017|         Hyderabad|   6000|
|    1030|          Thrissur|   2500|
|    1011|             Delhi|  44000|
|    1001|            Mumbai|  90000|
|    1024|            Kanpur|   4800|
|    1019|         Hyderabad|  45000|
|    1016|              Pune|   8000|
|    1002|             Delhi|  15000|
|    1015|             Surat|   5000|
+--------+------------------+-------+
only showing top 20 rows


**Demonstrate Transformations**

In [17]:
selected_df = df.select(
    "city",
    "revenue"
)

In [18]:
high_sales = df.filter(
    col("revenue") > 50000
)

In [19]:
city_revenue = df.groupBy(
    "city"
).sum("revenue")

In [20]:
df = df.withColumn(
    "discounted_revenue",
    col("revenue") * 0.95
)

**Demonstrate Actions**

In [21]:
df.show()

+--------+-------------+----------+-----------+--------+----------+----------+------------------+-----------+-------+------------------+
|order_id|customer_name|   product|   category|quantity|unit_price|order_date|              city|  sales_rep|revenue|discounted_revenue|
+--------+-------------+----------+-----------+--------+----------+----------+------------------+-----------+-------+------------------+
|    1014|   Arjun Nair|Headphones|Electronics|       3|      3500|21-01-2024|             Kochi|Anil Sharma|  10500|            9975.0|
|    1029| Sanjay Dubey|   USB Hub|Accessories|      10|       600|2024-02-20|         Allahabad| Sunita Rao|   6000|            5700.0|
|    1021| Nisha Kapoor|Headphones|Electronics|       2|      3500|2024-02-05|        Chandigarh|Anil Sharma|   7000|            6650.0|
|    1012|   SURESH RAO|   USB Hub|Accessories|       8|       600|2024-01-20|         Bangalore| Sunita Rao|   4800|            4560.0|
|    1009|   Ananya Das|    Laptop|Electr

In [22]:
df.count()

30

In [23]:
df.first()

Row(order_id=1014, customer_name='Arjun Nair', product='Headphones', category='Electronics', quantity=3, unit_price=3500, order_date='21-01-2024', city='Kochi', sales_rep='Anil Sharma', revenue=10500, discounted_revenue=9975.0)

In [24]:
df.collect()

[Row(order_id=1014, customer_name='Arjun Nair', product='Headphones', category='Electronics', quantity=3, unit_price=3500, order_date='21-01-2024', city='Kochi', sales_rep='Anil Sharma', revenue=10500, discounted_revenue=9975.0),
 Row(order_id=1029, customer_name='Sanjay Dubey', product='USB Hub', category='Accessories', quantity=10, unit_price=600, order_date='2024-02-20', city='Allahabad', sales_rep='Sunita Rao', revenue=6000, discounted_revenue=5700.0),
 Row(order_id=1021, customer_name='Nisha Kapoor', product='Headphones', category='Electronics', quantity=2, unit_price=3500, order_date='2024-02-05', city='Chandigarh', sales_rep='Anil Sharma', revenue=7000, discounted_revenue=6650.0),
 Row(order_id=1012, customer_name='SURESH RAO', product='USB Hub', category='Accessories', quantity=8, unit_price=600, order_date='2024-01-20', city='Bangalore', sales_rep='Sunita Rao', revenue=4800, discounted_revenue=4560.0),
 Row(order_id=1009, customer_name='Ananya Das', product='Laptop', category=

**Top 5 Cities by Revenue**

In [25]:
from pyspark.sql.functions import sum

top5 = df.groupBy(
    "city"
).agg(
    sum("revenue").alias(
        "total_revenue"
    )
).orderBy(
    "total_revenue",
    ascending=False
)

In [26]:
top5.show(5)

+------------------+-------------+
|              city|total_revenue|
+------------------+-------------+
|            Mumbai|       186000|
|Thiruvananthapuram|       135000|
|           Kolkata|        94800|
|             Delhi|        81000|
|              Pune|        61000|
+------------------+-------------+
only showing top 5 rows


**Save Top 5 Cities**

In [27]:
top5.toPandas().to_csv(
    "top5_city_revenue.csv",
    index=False
)

**CSV vs Parquet Comparison**

In [28]:
df.write.mode(
    "overwrite"
).parquet(
    "cleaned_sales_data.parquet"
)

In [29]:
import time

start = time.time()

spark.read.csv(
    "messy_sales_data.csv",
    header=True,
    inferSchema=True
).count()

csv_time = time.time() - start

In [30]:
start = time.time()

spark.read.parquet(
    "cleaned_sales_data.parquet"
).count()

parquet_time = time.time() - start

In [31]:
print("CSV Read Time:", csv_time)

print("Parquet Read Time:", parquet_time)

CSV Read Time: 0.6506049633026123
Parquet Read Time: 1.115370750427246


**Check File Sizes**

In [32]:
import os

csv_size = os.path.getsize(
    "messy_sales_data.csv"
)

print(
    "CSV Size:",
    csv_size,
    "bytes"
)

CSV Size: 2289 bytes


In [33]:
import os

parquet_folder = "cleaned_sales_data.parquet"

total_size = 0

for root, dirs, files in os.walk(parquet_folder):
    for file in files:
        total_size += os.path.getsize(
            os.path.join(root, file)
        )

print(
    "Parquet Size:",
    total_size,
    "bytes"
)

Parquet Size: 4934 bytes
